In [1]:
#0 라이브러리 & 데이터
import os, glob, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# Kaggle 경로 자동 탐색 (로컬이면 ../data)
cands = glob.glob('/kaggle/input/**/train.csv', recursive=True)
DATA = os.path.dirname(cands[0]) if cands else '../data'
train_raw = pd.read_csv(f'{DATA}/train.csv')
test_raw = pd.read_csv(f'{DATA}/test.csv')
print(DATA, train_raw.shape, test_raw.shape)

/kaggle/input/competitions/house-prices-advanced-regression-techniques (1460, 81) (1459, 80)


In [2]:
#1 전처리 함수 (옵션으로 v1 ↔ v2 비교 가능)
from scipy.stats import skew, boxcox_normmax
from scipy.special import boxcox1p

QUAL = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

def build(train, test, boxcox=True, more_ord=True, interact=True, drop_rare=True):
    # 1. 이상치 제거 + 타깃 로그 변환
    train = train[~train['Id'].isin([524, 1299])].reset_index(drop=True)
    y = np.log1p(train['SalePrice'])
    n = len(train)
    a = pd.concat([train.drop(columns=['Id', 'SalePrice']),
                   test.drop(columns=['Id'])]).reset_index(drop=True)

    # 2. 결측치 (v1과 동일)
    for c in ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
              'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
              'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']:
        a[c] = a[c].fillna('None')
    for c in ['GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
              'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']:
        a[c] = a[c].fillna(0)
    a['GarageYrBlt'] = a['GarageYrBlt'].fillna(a['YearBuilt'])
    a.loc[a['GarageYrBlt'] > 2010, 'GarageYrBlt'] = a['YearBuilt']   # test의 2207년 오타
    a['LotFrontage'] = a.groupby('Neighborhood')['LotFrontage'].transform(lambda s: s.fillna(s.median()))
    a['Functional'] = a['Functional'].fillna('Typ')
    for c in ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']:
        a[c] = a[c].fillna(a[:n][c].mode()[0])
    a = a.drop(columns=['Utilities'])
    for c in ['MSSubClass', 'MoSold', 'YrSold']:
        a[c] = a[c].astype(str)

    # 3. 파생 변수 (v1과 동일)
    ys = a['YrSold'].astype(int)
    a['TotalSF'] = a['TotalBsmtSF'] + a['1stFlrSF'] + a['2ndFlrSF']
    a['TotalBath'] = a['FullBath'] + 0.5 * a['HalfBath'] + a['BsmtFullBath'] + 0.5 * a['BsmtHalfBath']
    a['HouseAge'] = (ys - a['YearBuilt']).clip(lower=0)
    a['RemodAge'] = (ys - a['YearRemodAdd']).clip(lower=0)
    a['TotalPorchSF'] = (a['OpenPorchSF'] + a['EnclosedPorch'] + a['3SsnPorch']
                         + a['ScreenPorch'] + a['WoodDeckSF'])
    for c in ['PoolArea', '2ndFlrSF', 'GarageArea', 'TotalBsmtSF', 'MasVnrArea', 'Fireplaces',
              'MiscVal', '3SsnPorch', 'LowQualFinSF', 'BsmtFinSF2', 'ScreenPorch', 'EnclosedPorch',
              'WoodDeckSF', 'OpenPorchSF']:
        a['Has' + c] = (a[c] > 0).astype(int)

    # 4. 순서형 인코딩
    for c in ['ExterQual', 'KitchenQual', 'BsmtQual', 'FireplaceQu', 'HeatingQC']:
        a[c] = a[c].map(QUAL)
    a['GarageFinish'] = a['GarageFinish'].map({'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3})
    a['BsmtExposure'] = a['BsmtExposure'].map({'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4})
    if more_ord:   # [v2] 추가
        for c in ['GarageQual', 'GarageCond', 'BsmtCond', 'PoolQC']:
            a[c] = a[c].map(QUAL)
        a['Functional'] = a['Functional'].map({'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
                                               'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7})
        a['CentralAir'] = (a['CentralAir'] == 'Y').astype(int)
        a['PavedDrive'] = a['PavedDrive'].map({'N': 0, 'P': 1, 'Y': 2})

    # 5. 대부분 0인 컬럼은 유무만 남김 + 희귀 범주 묶기 (v1과 동일)
    a = a.drop(columns=['MiscVal', '3SsnPorch', 'LowQualFinSF', 'BsmtFinSF2',
                        'ScreenPorch', 'EnclosedPorch', 'PoolArea'])
    for c in ['Condition2', 'RoofMatl', 'Heating']:
        cnt = a[:n][c].value_counts()
        a[c] = a[c].where(a[c].isin(cnt[cnt >= 10].index), 'Other')

    # 6. 치우친 연속형 변환
    if boxcox:   # [v2] 왜도 0.5 이상인 연속형 전부 Box-Cox, λ는 train으로만 추정
        num = [c for c in a.select_dtypes(include='number').columns
               if a[c].nunique() > 15 and not c.startswith('Has')
               and c not in ('YearBuilt', 'YearRemodAdd', 'GarageYrBlt')]
        sk = a[:n][num].apply(lambda s: skew(s))
        for c in sk[sk.abs() > 0.5].index:
            if (a[c] < 0).any():
                continue
            try:
                lam = boxcox_normmax(a[:n][c] + 1, method='mle')
            except Exception:
                lam = 0.0   # 추정 실패하면 로그 변환과 같음
            a[c] = boxcox1p(a[c], lam)
    else:        # v1 방식
        for c in ['LotArea', 'LotFrontage', 'BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF',
                  '1stFlrSF', 'GrLivArea', 'TotalSF', 'MasVnrArea', 'TotalPorchSF']:
            a[c] = np.log1p(a[c])

    if interact:   # [v2] 기본 탑재
        a['Qual_SF'] = a['OverallQual'] * a['TotalSF']
        a['Qual_Liv'] = a['OverallQual'] * a['GrLivArea']

    # 7. 원-핫 + 희귀 더미 제거
    a = pd.get_dummies(a).astype(float)
    if drop_rare:   # [v2] train에서 5채 미만만 다른 값 → 과적합 재료
        diff = (a[:n] != a[:n].mode().iloc[0]).sum()
        a = a.drop(columns=diff[diff < 5].index)

    X, X_test = a[:n].reset_index(drop=True), a[n:].reset_index(drop=True)
    return X, X_test, y, train

X, X_test, y, train = build(train_raw, test_raw)
print(X.shape, X_test.shape, '결측:', X.isna().sum().sum() + X_test.isna().sum().sum())

(1458, 252) (1459, 252) 결측: 0


In [3]:
#2 검증 도구
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

kf = KFold(n_splits=5, shuffle=True, random_state=42)                    # OOF용
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)           # 피처 비교용 (15번)
rmse = lambda p: np.sqrt(((y.values - p) ** 2).mean())

def cv_scores(model, X_):
    return -cross_val_score(model, X_, y, cv=rkf, scoring='neg_root_mean_squared_error')

In [4]:
#3 변경 효과 (반복 교차검증 15번, 짝지어 비교)
from sklearn.linear_model import Lasso
from sklearn.svm import SVR

lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=50000))
svr = make_pipeline(RobustScaler(), SVR(C=15, epsilon=0.008, gamma=0.0005))

v1 = dict(boxcox=False, more_ord=False, interact=False, drop_rare=False)
X_v1 = build(train_raw, test_raw, **v1)[0]
base = {n: cv_scores(m, X_v1) for n, m in [('Lasso', lasso), ('SVR', svr)]}

rows = []
for name, change in [('Box-Cox', {'boxcox': True}), ('순서형 확대', {'more_ord': True}),
                     ('상호작용', {'interact': True}), ('희귀 더미 제거', {'drop_rare': True}),
                     ('v2 전체', {k: True for k in v1})]:
    Xc = build(train_raw, test_raw, **{**v1, **change})[0]
    row = {'변경': name, '컬럼 수': Xc.shape[1]}
    for n, m in [('Lasso', lasso), ('SVR', svr)]:
        s = cv_scores(m, Xc)
        row[n] = round(s.mean(), 4)
        row[n + ' 개선 횟수'] = f'{(s < base[n]).sum()}/15'
    rows.append(row)
print('v1 기준: Lasso', base['Lasso'].mean().round(4), '| SVR', base['SVR'].mean().round(4))
pd.DataFrame(rows)

v1 기준: Lasso 0.1117 | SVR 0.111


,변경,컬럼 수,Lasso,Lasso 개선 횟수,SVR,SVR 개선 횟수
0,Box-Cox,302,0.1111,9/15,0.1101,10/15
1,순서형 확대,276,0.1108,12/15,0.1103,12/15
2,상호작용,304,0.1114,15/15,0.1108,10/15
3,희귀 더미 제거,268,0.1117,6/15,0.1110,6/15
4,v2 전체,252,0.1093,15/15,0.1088,14/15


In [5]:
#4 구조 모델 v3 (v1 6-1 코드를 클래스로 정리, 내용 동일)
from scipy.optimize import least_squares

qmap = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
SALE = ['Abnorml', 'Family', 'Alloca', 'Partial', 'AdjLand']
ZONE = ['C (all)', 'RM', 'RH', 'FV']

def prep_struct(d):
    d = d.copy()
    for c in ['TotalBsmtSF', 'GarageArea', 'BsmtFullBath', 'BsmtHalfBath']:
        d[c] = d[c].fillna(0)
    ys = d['YrSold'].astype(int)
    d['age'] = (ys - d['YearBuilt']).clip(lower=0)
    d['remod_age'] = (ys - d['YearRemodAdd']).clip(lower=0)
    d['MSZoning'] = d['MSZoning'].fillna('RL')
    d['KQ'] = d['KitchenQual'].map(qmap).fillna(3)
    d['bath'] = d['FullBath'] + 0.5 * d['HalfBath'] + d['BsmtFullBath'] + 0.5 * d['BsmtHalfBath']
    d['porch'] = d[['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', '3SsnPorch']].sum(axis=1)
    d['air'] = (d['CentralAir'] == 'Y').astype(float)
    return d

class StructModel:
    """땅값(동네별) + 건물값(동네별 단가 × 면적 × 품질 × 감가) + 부대시설"""
    def __init__(self, neighborhoods, lam=0.3):
        self.idx = {n: i for i, n in enumerate(neighborhoods)}
        self.K, self.lam = len(neighborhoods), lam

    def _predict(self, th, d):
        K = self.K; land, bld = th[:K], th[K:2*K]; o = 2 * K
        b_lot, w_bs, w_gar, g_q, g_c, d_age, d_rem = th[o:o+7]; o += 7
        s_mult = th[o:o+5]; o += 5
        z_mult = th[o:o+4]; o += 4
        e_bath, e_fp, e_porch, g_k, g_air = th[o:o+5]
        i = d['Neighborhood'].map(self.idx).values
        land_val = np.exp(land[i]) * (d['LotArea'].values / 10000) ** b_lot
        area = d['GrLivArea'].values + w_bs * d['TotalBsmtSF'].values + w_gar * d['GarageArea'].values
        quality = np.exp(g_q * (d['OverallQual'].values - 5) + g_c * (d['OverallCond'].values - 5))
        dep = np.exp(-d_age * d['age'].values - d_rem * d['remod_age'].values)
        total = land_val + np.exp(bld[i]) * area * quality * dep / 1000
        total = total * np.exp(g_k * (d['KQ'].values - 3) + g_air * (d['air'].values - 1)) \
              + e_bath * d['bath'].values + e_fp * d['Fireplaces'].values + e_porch * d['porch'].values / 100
        adj = sum(s_mult[j] * (d['SaleCondition'].values == c) for j, c in enumerate(SALE)) \
            + sum(z_mult[j] * (d['MSZoning'].values == c) for j, c in enumerate(ZONE))
        return np.log1p(np.clip(total, 1, None) * np.exp(adj) * 1000)

    def fit(self, d, yv):
        K = self.K
        th0 = np.r_[np.full(K, np.log(30)), np.full(K, np.log(80)),
                    0.5, 0.4, 0.3, 0.15, 0.03, 0.005, 0.002, np.zeros(9), 3, 3, 1, 0.05, 0.05]
        def residual(th):
            pen = 0.1 * np.sqrt(self.lam) * np.r_[th[:K] - th[:K].mean(), th[K:2*K] - th[K:2*K].mean()]
            return np.r_[self._predict(th, d) - yv, pen]
        self.th = least_squares(residual, th0, max_nfev=5000).x
        return self

    def predict(self, d):
        return self._predict(self.th, d)

S_train, S_test = prep_struct(train), prep_struct(test_raw)
NBS = sorted(S_train['Neighborhood'].unique())

In [6]:
#5 베이스 모델
from xgboost import XGBRegressor

abn = (train['SaleCondition'] != 'Normal').values
w_abn = np.where(abn, 0.5, 1.0)   # v1 5-16: 비정상 거래 가중치 0.5 (Lasso만)

def make_xgb(seed=42):
    return XGBRegressor(learning_rate=0.02, n_estimators=2500, max_depth=3, subsample=0.7,
                        colsample_bytree=0.2, min_child_weight=3, random_state=seed, n_jobs=-1)

base_models = {
    'Lasso_w': (lasso, {'lasso__sample_weight': w_abn}),
    'XGBoost': (make_xgb(), None),
    'SVR':     (svr, None),
}

In [7]:
#6 OOF 예측 (같은 분할 kf)
oof = pd.DataFrame(index=range(len(y)))
for name, (m, p) in base_models.items():
    t = time.time()
    oof[name] = cross_val_predict(m, X, y, cv=kf, params=p)
    print(f'{name:8s} {rmse(oof[name]):.4f}  ({time.time() - t:.0f}초)')

oof['구조v3'] = 0.0
for tr_i, va_i in kf.split(S_train):
    sm = StructModel(NBS).fit(S_train.iloc[tr_i], y.values[tr_i])
    oof.loc[va_i, '구조v3'] = sm.predict(S_train.iloc[va_i])
print(f'구조v3   {rmse(oof["구조v3"]):.4f}')

oof.sub(y.values, axis=0).corr().round(2)   # 오차 상관

Lasso_w  0.1092  (1초)
XGBoost  0.1119  (20초)
SVR      0.1094  (2초)
구조v3   0.1146


,Lasso_w,XGBoost,SVR,구조v3
Lasso_w,1.00,0.88,0.94,0.84
XGBoost,0.88,1.00,0.87,0.80
SVR,0.94,0.87,1.00,0.79
구조v3,0.84,0.80,0.79,1.00


In [8]:
#7 스태킹 vs 단순 평균 (스태킹 분할 10가지)
from sklearn.linear_model import LinearRegression

cols = ['Lasso_w', 'XGBoost', 'SVR', '구조v3']
meta = LinearRegression(positive=True)

stack_s = [rmse(cross_val_predict(meta, oof[cols], y, cv=KFold(5, shuffle=True, random_state=s)))
           for s in range(10)]
print(f'스태킹    {np.mean(stack_s):.4f} (±{np.std(stack_s):.4f}, 10가지 분할)')
print(f'단순 평균 {rmse(oof[cols].mean(axis=1)):.4f}')

meta.fit(oof[cols], y)
print('스태킹 가중치:', dict(zip(cols, meta.coef_.round(2))),
      '| 합', meta.coef_.sum().round(2), '| 절편', round(meta.intercept_, 3))

스태킹    0.1049 (±0.0002, 10가지 분할)
단순 평균 0.1048
스태킹 가중치: {'Lasso_w': np.float64(0.04), 'XGBoost': np.float64(0.23), 'SVR': np.float64(0.45), '구조v3': np.float64(0.3)} | 합 1.02 | 절편 -0.286


In [9]:
#8 최종 학습 + 제출 파일 2개
test_preds = pd.DataFrame({
    'Lasso_w': lasso.fit(X, y, lasso__sample_weight=w_abn).predict(X_test),
    'XGBoost': np.mean([make_xgb(s).fit(X, y).predict(X_test) for s in [42, 7, 2026]], axis=0),
    'SVR':     svr.fit(X, y).predict(X_test),
    '구조v3':  StructModel(NBS).fit(S_train, y.values).predict(S_test),
})

# Id 2550 규칙
twin_log = np.log1p(train_raw.loc[train_raw['Id'].isin([524, 1299]), 'SalePrice']).mean()
special = ((test_raw['GrLivArea'] > 4000) & (test_raw['SaleCondition'] == 'Partial')).values

outputs = {'submission_stack.csv': meta.predict(test_preds[cols]),
           'submission_blend.csv': test_preds[cols].mean(axis=1).values}
for fname, pred_log in outputs.items():
    pred_log = pred_log.copy()
    pred_log[special] = twin_log
    sub = pd.DataFrame({'Id': test_raw['Id'], 'SalePrice': np.expm1(pred_log)})
    sub.to_csv(fname, index=False)
    print(fname, sub.shape, '| 2550:', round(sub.loc[sub['Id'] == 2550, 'SalePrice'].iloc[0]))

# 모델 간 test 예측 상관 (너무 낮은 모델이 있으면 이상 신호)
test_preds.corr().round(3)

submission_stack.csv (1459, 2) | 2550: 171930
submission_blend.csv (1459, 2) | 2550: 171930


,Lasso_w,XGBoost,SVR,구조v3
Lasso_w,1.000,0.991,0.995,0.986
XGBoost,0.991,1.000,0.989,0.984
SVR,0.995,0.989,1.000,0.981
구조v3,0.986,0.984,0.981,1.000


In [10]:
#9 점검: test 예측 분포가 train과 비슷한지
print('train  log 가격:', y.describe()[['mean', 'std', 'min', 'max']].round(3).to_dict())
for f in outputs:
    p = np.log1p(pd.read_csv(f)['SalePrice'])
    print(f'{f:22s}', p.describe()[['mean', 'std', 'min', 'max']].round(3).to_dict())

train  log 가격: {'mean': 12.024, 'std': 0.4, 'min': 10.46, 'max': 13.534}
submission_stack.csv   {'mean': 12.012, 'std': 0.4, 'min': 10.611, 'max': 13.262}
submission_blend.csv   {'mean': 12.012, 'std': 0.393, 'min': 10.622, 'max': 13.211}
